# test-2 scaled dot product attention

## softmax

In [2]:
from typing import Optional

import torch
import torch.nn as nn

In [3]:
def softmax(x: torch.Tensor, dim: int, subtract_max: bool = True) -> torch.Tensor:
    '''
    Use the trick of subtracting the maximum value in
    the i-th dimension from all elements of the i-th dimension to avoid numerical stability issues.
    '''
    x_max = x.max(dim=dim, keepdim=True).values
    if subtract_max:
        x_ = x - x_max
    else:
        x_ = x
    x_exp = torch.exp(x_)
    x_exp_sum = x_exp.sum(dim=-1, keepdim=True)#.unsqueeze(-1)
    x_exp = x_exp/x_exp_sum
    return x_exp
    


In [4]:
x = torch.randn(2, 4)
softmax(x, dim=1)

tensor([[0.2597, 0.1909, 0.4174, 0.1320],
        [0.1286, 0.1619, 0.5794, 0.1301]])

In [5]:
# test 'softmax operation is invariant to adding any constant c to all inputs'ArithmeticError
out1 = softmax(x, dim=1, subtract_max=True)
out2 = softmax(x, dim=1, subtract_max=False)

In [6]:
print(out1[0])
print(out2[0])

tensor([0.2597, 0.1909, 0.4174, 0.1320])
tensor([0.2597, 0.1909, 0.4174, 0.1320])


## sdpa

In [7]:
# mask
def mask(batch_size, n_queries, n_keys):
    torch.manual_seed(5)
    return torch.randn(batch_size, n_queries, n_keys) > 0.5

In [8]:
batch_size = 2
n_queries = 3
n_keys = 3

attn_mask = mask(batch_size, n_queries, n_keys)

In [9]:
attn_mask

tensor([[[ True,  True, False],
         [False, False, False],
         [False, False,  True]],

        [[False, False, False],
         [ True, False, False],
         [ True,  True, False]]])

In [14]:
mask_tensor = torch.where(attn_mask, torch.tensor(0.0), torch.tensor(float('-inf')))
mask_tensor

tensor([[[0., 0., -inf],
         [-inf, -inf, -inf],
         [-inf, -inf, 0.]],

        [[-inf, -inf, -inf],
         [0., -inf, -inf],
         [0., 0., -inf]]])

In [16]:
x = torch.randn(batch_size, n_queries, n_keys)
x

tensor([[[ 1.6980, -0.1792,  0.1730],
         [ 0.1668, -1.1372, -0.8208],
         [ 0.9895,  0.5197, -0.6861]],

        [[-0.3714,  0.9906,  0.7426],
         [ 1.2702,  1.3227,  1.5585],
         [-0.3577, -0.2506, -2.6642]]])

In [18]:
## causal attention - torch.triu (triangle upper)
# query i should attend positions j<=i
causal_mask = torch.ones(3,3)
causal_mask = torch.triu(causal_mask, diagonal=1)==0
mask_tensor = torch.where(causal_mask, torch.tensor(0.0), torch.tensor(float('-inf')))
mask_tensor

tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.]])

In [27]:
causal_mask

tensor([[ True, False, False],
        [ True,  True, False],
        [ True,  True,  True]])

In [19]:
x+ mask_tensor

tensor([[[ 1.6980,    -inf,    -inf],
         [ 0.1668, -1.1372,    -inf],
         [ 0.9895,  0.5197, -0.6861]],

        [[-0.3714,    -inf,    -inf],
         [ 1.2702,  1.3227,    -inf],
         [-0.3577, -0.2506, -2.6642]]])

In [26]:
x = torch.randn(8, 16)
n_head = 4
d_head = 4
x.view(8, n_head, d_head).shape

torch.Size([8, 4, 4])